# Postav si vlastního chatbota 🤖

**Letní škola AI · praktická část**

Na přednášce jsme skončili u AI agentů. Teď si jednoho — v malém — postavíš sám. Tvůj chatbot:

1. bude mít **osobnost**, kterou mu vymyslíš ty (třeba vševěd, který hádá, na co myslíš),
2. bude si **pamatovat**, o čem se spolu bavíte,
3. *(bonus)* bude si umět **zavolat nástroj** — třeba hodit kostkou.

Nebudeme trénovat žádnou neuronovou síť. Použijeme hotový velký jazykový model (LLM), který běží v cloudu, a budeme s ním komunikovat přes **API** — tedy stejně, jako to uvnitř dělá ChatGPT a všichni ostatní chatboti.



## 0 · Příprava: API klíč

Model poběží na službě **Groq** (pozor, nezaměnit s Grokem od xAI — Groq je firma vyrábějící čipy, na kterých modely běží extrémně rychle). Pro naše účely je zdarma, stačí se zaregistrovat:

1. Otevři [console.groq.com](https://console.groq.com) a přihlas se (jde to Google účtem).
2. V menu vlevo najdi **API Keys** → **Create API Key**.
3. Pojmenuj ho třeba `letni-skola` a klíč si **hned zkopíruj** — zobrazí se jen jednou.

> ⚠️ **API klíč je jako heslo.** Nikomu ho neposílej, nedávej ho přímo do kódu a nikam ho nezveřejňuj. Proto ho v další buňce zadáme přes skryté pole, ne napevno do programu.

Teď nainstalujeme knihovnu a připravíme spojení:

In [ ]:
%pip install --quiet openai

from getpass import getpass
from openai import OpenAI

GROQ_API_KLIC = getpass("Vlož svůj Groq API klíč a stiskni Enter (text nebude vidět): ")

klient = OpenAI(
    api_key=GROQ_API_KLIC,
    base_url="https://api.groq.com/openai/v1",  # Groq "mluví" stejným API jako OpenAI
)

MODEL = "llama-3.3-70b-versatile"  # aktuální nabídka modelů: https://console.groq.com/docs/models

print("Hotovo, spojení je připravené.")

---
# Dílky skládačky

Než chatbota složíme, prohlédneme si jednotlivé dílky. Každý je pár řádků kódu.

## Dílek 1 · REPL — program, který se pořád ptá

**REPL** = *Read → Eval → Print → Loop*: přečti vstup, zpracuj, vypiš, opakuj. Přesně takhle funguje každé chatovací okno. Tady je nejhloupější možný chatbot — papoušek (žádná AI, jen smyčka):

In [ ]:
print("Papoušek běží! Ukončíš ho slovem 'konec'.")

while True:
    vstup = input("Ty: ")
    if vstup.lower() == "konec":
        print("Papoušek: Tak zase příště!")
        break
    print("Papoušek:", vstup)

Nic víc REPL není: `while True`, `input()`, `print()` a podmínka na ukončení. Zapamatuj si ten tvar — na konci do něj jen místo papouška dosadíme jazykový model.

## Dílek 2 · Jeden dotaz přes API

Teď pošleme **jednu zprávu** modelu. Dotaz má vždy stejný tvar: seznam zpráv (`messages`), kde každá zpráva má **roli** (`"user"` = člověk) a **obsah**:

In [ ]:
odpoved = klient.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Ahoj! Vysvětli mi jednou větou, co je to API."},
    ],
)

print(odpoved.choices[0].message.content)

Zkus změnit text dotazu a spustit buňku znovu. Odpověď přijde za zlomek sekundy — proto používáme Groq.

> 🔍 Chceš vidět, co všechno API vrací? Spusť v nové buňce `print(odpoved)` — uvidíš i počty tokenů (o těch byla řeč na přednášce).

## Dílek 3 · Osobnost přes system prompt

Kromě role `"user"` existuje role `"system"`. Zpráva s touto rolí je **instrukce pro model** — kdo je, jak se má chovat, co smí a nesmí. Uživatel ji nevidí, ale model se jí řídí. Říká se jí **system prompt**.

In [ ]:
OSOBNOST = """Jsi Vševěd — tajemná věštkyně z pouti.
Hráč si myslí nějakou věc a ty se ji snažíš uhodnout.
Pokládej otázky, na které se dá odpovědět ano/ne. Vždy jen jednu otázku najednou.
Když si jsi jistá, vyslov svůj tip. Mluv vznešeně a trochu záhadně."""

odpoved = klient.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": OSOBNOST},
        {"role": "user", "content": "Ahoj, myslím si jednu věc. Do toho!"},
    ],
)

print(odpoved.choices[0].message.content)

Všimni si, že osobnost je jen **text v proměnné**. Když ho přelepíš jiným textem, máš okamžitě jiného bota — pirát, kvízmistr, Karel IV., zarputilý robot… Zkus to: přepiš `OSOBNOST` a spusť buňku znovu.

## Dílek 4 · Model si nic nepamatuje

Teď důležitá věc, na které chatboti stojí a padají. Každé volání API začíná **od nuly** — model si z minulého dotazu nepamatuje vůbec nic. Vyzkoušej:

In [ ]:
# První dotaz: představíme se.
klient.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Jmenuji se Kuba a je mi 16 let."}],
)

# Druhý dotaz: zeptáme se na jméno.
odpoved = klient.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Jak se jmenuji?"}],
)

print(odpoved.choices[0].message.content)

Model netuší. **Paměť si musíme vyrobit sami** — a dělá se to překvapivě jednoduše: všechny zprávy (naše i modelu) si ukládáme do seznamu a při každém dotazu posíláme **celou historii znovu**. Odpovědi modelu se do historie ukládají s rolí `"assistant"`.

In [ ]:
historie = [
    {"role": "system", "content": "Jsi stručný a přátelský pomocník."},
]

def zeptej_se(text):
    """Přidá dotaz do historie, pošle modelu VŠECHNO a jeho odpověď si taky uloží."""
    historie.append({"role": "user", "content": text})
    odpoved = klient.chat.completions.create(model=MODEL, messages=historie)
    zprava = odpoved.choices[0].message.content
    historie.append({"role": "assistant", "content": zprava})
    return zprava

print(zeptej_se("Jmenuji se Kuba a je mi 16 let."))
print()
print(zeptej_se("Jak se jmenuji?"))

Teď už si jméno pamatuje! Když si spustíš `print(historie)`, uvidíš přesně to, co model při druhém dotazu dostal: system prompt + celý dosavadní rozhovor. Tohle je i důvod, proč mají modely omezené **kontextové okno** — celý rozhovor se musí pokaždé vejít dovnitř.

---
# Hlavní úkol · Slož si vlastního chatbota

Máš všechny dílky: **REPL** (dílek 1) + **API** (dílek 2) + **osobnost** (dílek 3) + **paměť** (dílek 4). Teď je poskládej dohromady:

1. **Vymysli scénář.** Pár nápadů na rozjezd:

   | Scénář | System prompt v kostce |
   |---|---|
   | 🔮 **Vševěd** | hádá, na co myslíš, přes ano/ne otázky |
   | 🏆 **Kvízmistr** | dává otázky z tématu, které si vybereš, a počítá body |
   | 👑 **Karel IV.** | odpovídá jako on, o ničem po roce 1378 neví |
   | 🤖 **Zarputilý strážce** | zná tajné heslo a nesmí ho prozradit — zkus ho z něj vymámit! |
   | 🧭 **Průvodce po Praze** | doporučuje výlety, mluví jako nadšený kamarád |

2. **Napiš system prompt** — kdo bot je, co je jeho úkol, jaká má pravidla (jak dlouze odpovídá, co nesmí…).
3. **Doplň chybějící řádky** v kostře níže (všechno už jsi viděl v dílcích 2 a 4).
4. **Otestuj** — a pak si vyměň místo se sousedem a zkuste si navzájem bota „rozbít".

In [ ]:
# ================== TVŮJ CHATBOT ==================

OSOBNOST = """
DOPLŇ: Kdo je tvůj bot? Co je jeho úkol? Jaká má pravidla?
"""

historie = [{"role": "system", "content": OSOBNOST}]

print("Chatbot běží! Ukončíš ho slovem 'konec'.")

# Ve smyčce:

    # DOPLŇ 1: přidej vstup do historie jako zprávu s rolí "user"

    # DOPLŇ 2: zavolej model s celou historií

    # DOPLŇ 3: vytáhni text odpovědi a ulož ho do historie s rolí "assistant"

    # DOPLŇ 4: vypiš odpověď


> 💡 **Když bot zlobí:**
> * *Vypadává z role* → přitvrď pravidla v system promptu („Nikdy nevystupuj z role. Když se tě zeptají na něco mimo hru, odpověz v duchu své postavy.").
> * *Odpovídá moc dlouze* → přidej „Odpovídej nejvýše dvěma větami."
> * *Chyba `429 rate limit`* → posíláš dotazy moc rychle za sebou, chvilku počkej a pokračuj.

---
# Bonus · Dej botovi nástroj 🎲

Na přednášce padlo, že **agent = LLM + nástroje ve smyčce**. Jazykový model sám neumí ani vygenerovat opravdové náhodné číslo — jen předpovídá tokeny, takže si ho *vymyslí*. (Vyzkoušej: požádej svého bota, ať hodí kostkou. Číslo napíše, ale férová náhoda to není.)

Řešení: napíšeme obyčejnou pythonovskou funkci a **popíšeme ji modelu**. Model pak může místo odpovědi říct „zavolej za mě `hod_kostkou(20)`" — my funkci spustíme, výsledek mu pošleme zpět a on teprve odpoví. Tomu se říká **tool calling** a funguje úplně stejně u velkých agentů (hledání na webu, spouštění kódu, čtení souborů…).

In [ ]:
import json
import random

# 1) Obyčejná funkce — tohle je celý náš "nástroj".
def hod_kostkou(pocet_sten):
    """Vrátí náhodné celé číslo od 1 do pocet_sten."""
    return random.randint(1, int(pocet_sten))

# 2) Popis nástroje pro model: jak se jmenuje, k čemu je, jaké má parametry.
#    Model čte jen tenhle popis — samotný kód funkce nikdy nevidí.
NASTROJE = [
    {
        "type": "function",
        "function": {
            "name": "hod_kostkou",
            "description": "Hodí férovou kostkou a vrátí náhodné číslo od 1 do pocet_sten.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pocet_sten": {
                        "type": "integer",
                        "description": "Počet stěn kostky, například 6 nebo 20.",
                    }
                },
                "required": ["pocet_sten"],
            },
        },
    }
]

In [ ]:
OSOBNOST = """Jsi Pán jeskyně ve fantasy hře na hrdiny. Stručně (2–4 věty) popisuješ,
co se děje, a hráč říká, co dělá jeho postava. Kdykoli hráč zkusí něco riskantního,
hoď dvacetistěnnou kostkou nástrojem hod_kostkou: 1–7 neúspěch, 8–14 úspěch se
zádrhelem, 15–20 úspěch. Výsledek hodu vždy prozraď. Začni krátkým úvodem do příběhu."""

historie = [{"role": "system", "content": OSOBNOST}]

def zeptej_se_s_nastroji(text):
    historie.append({"role": "user", "content": text})
    while True:  # opakujeme, dokud model chce volat nástroje
        odpoved = klient.chat.completions.create(model=MODEL, messages=historie, tools=NASTROJE)
        zprava = odpoved.choices[0].message
        # Explicitně vytvoříme slovník pro historii, abychom se vyhnuli nepodporovaným polím jako 'annotations'.
        message_to_add_to_history = {
            "role": zprava.role,
            "content": zprava.content
        }
        if zprava.tool_calls:
            # Pokud jsou přítomny tool_calls, musíme je také přidat a převést na slovníky.
            message_to_add_to_history["tool_calls"] = [tc.model_dump(exclude_none=True) for tc in zprava.tool_calls]
        historie.append(message_to_add_to_history)

        if not zprava.tool_calls:          # model už nástroj nechce -> máme odpověď
            return zprava.content
        for volani in zprava.tool_calls:   # model chce hodit kostkou (klidně vícekrát)
            argumenty = json.loads(volani.function.arguments)
            vysledek = hod_kostkou(**argumenty)
            print(f"   🎲 hod_kostkou({argumenty['pocet_sten']}) → {vysledek}")
            historie.append({"role": "tool", "tool_call_id": volani.id, "content": str(vysledek)})

print("Hra běží! Ukončíš ji slovem 'konec'.")
print("PJ:", zeptej_se_s_nastroji("Ahoj, jsem připraven. Kde se nacházím?"))

while True:
    vstup = input("Ty: ")
    if vstup.lower() == "konec":
        break
    print("PJ:", zeptej_se_s_nastroji(vstup))

Sleduj řádky s 🎲 — to je okamžik, kdy model **sám rozhodl** použít nástroj, a náhodu dodal tvůj Python, ne model.

**Nápady na vlastní nástroje** (stačí napsat funkci a přidat její popis do `NASTROJE`):
* `aktualni_cas()` — vrátí datum a čas (`datetime.now()`); model ho sám od sebe nezná!
* `tajne_cislo()` — na začátku hry vylosuje číslo 1–100 a bot pak vede hru „hádej číslo, řeknu ti víc/míň".
* `spocitej(vyraz)` — spolehlivá kalkulačka, aby bot nepočítal „od oka".

---
# Referenční řešení hlavního úkolu

*(Nekoukej, dokud sis to nezkusil sám!)*

In [ ]:
OSOBNOST = """Jsi Vševěd — tajemná věštkyně z pouti.
Hráč si myslí nějakou věc a ty se ji snažíš uhodnout.
Pokládej otázky, na které se dá odpovědět ano/ne. Vždy jen jednu otázku najednou.
Když si jsi jistá, vyslov svůj tip. Mluv vznešeně, záhadně a stručně.
Nikdy nevystupuj z role."""

historie = [{"role": "system", "content": OSOBNOST}]

print("Chatbot běží! Ukončíš ho slovem 'konec'.")

while True:
    vstup = input("Ty: ")
    if vstup.lower() == "konec":
        break

    historie.append({"role": "user", "content": vstup})                       # DOPLŇ 1
    odpoved = klient.chat.completions.create(model=MODEL, messages=historie)  # DOPLŇ 2
    zprava = odpoved.choices[0].message.content
    historie.append({"role": "assistant", "content": zprava})                 # DOPLŇ 3
    print("Bot:", zprava)                                                     # DOPLŇ 4

---
# Co sis dnes postavil

* **REPL** — smyčka čti → zpracuj → vypiš, základ každého chatu.
* **API dotaz** — seznam zpráv s rolemi `system` / `user` / `assistant`.
* **Osobnost** — jen text v system promptu; přelepíš text, máš jiného bota.
* **Paměť** — žádná magie: celá historie se posílá znovu s každým dotazem.
* **Nástroje** — model si řekne o funkci, tvůj kód ji spustí a vrátí mu výsledek.

Tohle je — doopravdy, bez zjednodušování — jádro toho, jak fungují ChatGPT, Claude i AI agenti. Všechno ostatní je škálování a ladění detailů.